# ALLaM-7B on the wider fully diacritized set (sft-allam-tashkeel-v3)
9,492 poems (76,318 abyat: exact or checker score >= 0.90; poems with >= 70% passing abyat kept whole), 1 epoch. Meter is enforced at decoding time, so the filter is looser and poems stay whole. Same recipe as allam_full_v1_qlora otherwise. Attach the dataset `arabic-poet-sft-tashkeel-v3`.

# Arabic Poet: full QLoRA of ALLaM-7B on Kaggle (free T4)

Same recipe as the planned Mac run (`scripts/full_training.py`), so the adapter converts back to MLX with
`scripts/convert_peft_lora.py` and is evaluated on the Mac exactly like the pilot:

- base: `humain-ai/ALLaM-7B-Instruct-preview` @ `a28dd1e67420cde72d3629c8633a974cf7d9c366` (4-bit NF4)
- LoRA rank 16, alpha 320 (= MLX scale 20), all linear layers of layers 16-31 (= MLX `num_layers: 16`)
- loss on the poem only (prompt masked), effective batch 8, 1 epoch, AdamW, peak LR 4e-5, 100 warmup steps, cosine to 10%
- data: `sft-allam-full-v1` (13,080 train / 1,513 valid / 1,523 test), public-domain poems from Arabic Wikisource

Run with **Save Version -> Save & Run All (Commit)**, accelerator **GPU T4 x2 or P100**, internet **on**. You can close the
browser; the run continues (max 12 h) and the adapter appears under *Output*. A time guard saves the adapter before the limit.


In [ ]:
%%capture
!pip install -q unsloth


In [ ]:
import glob, json, math, os, time, torch
START = time.time()
TIME_LIMIT_H = 11.0          # Kaggle commits stop at 12 h: save before that
REPO = "humain-ai/ALLaM-7B-Instruct-preview"
REV = "a28dd1e67420cde72d3629c8633a974cf7d9c366"
OUT = "/kaggle/working"
data_dir = os.path.dirname(glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0])
print("data:", data_dir, "| GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

def read(split):
    with open(f"{data_dir}/{split}.jsonl", encoding="utf-8") as f:
        return [json.loads(l)["messages"] for l in f if l.strip()]
train_msgs, valid_msgs, test_msgs = read("train"), read("valid"), read("test")
print(len(train_msgs), len(valid_msgs), len(test_msgs))


In [ ]:
from huggingface_hub import snapshot_download
base_dir = snapshot_download(REPO, revision=REV)   # pinned revision, public (ungated) model
USE_UNSLOTH = True
try:
    from unsloth import FastLanguageModel
    model, tok = FastLanguageModel.from_pretrained(base_dir, max_seq_length=1024, dtype=None, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=320, lora_dropout=0.0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        layers_to_transform=list(range(16, 32)), use_gradient_checkpointing="unsloth", random_state=0)
except Exception as exc:   # fallback: plain transformers + bitsandbytes + PEFT (slower, same recipe)
    print("Unsloth unavailable, falling back to PEFT:", exc)
    USE_UNSLOTH = False
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                             bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16)
    tok = AutoTokenizer.from_pretrained(base_dir)
    model = AutoModelForCausalLM.from_pretrained(base_dir, quantization_config=bnb, device_map={"": 0})
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=320, lora_dropout=0.0, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        layers_to_transform=list(range(16, 32))))
if tok.pad_token is None:
    tok.pad_token = tok.unk_token or tok.eos_token
model.print_trainable_parameters()


In [ ]:
# Prompt masking exactly like MLX-LM mask_prompt: loss only on the assistant reply.
def encode(msgs, max_len=1024):
    full = tok.apply_chat_template(msgs, tokenize=False)
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False)
    ids = tok(full, add_special_tokens=False)["input_ids"]
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    assert ids[:len(p_ids)] == p_ids, "prompt is not a token prefix of the full text"
    if len(ids) > max_len:
        return None
    return {"input_ids": ids, "attention_mask": [1] * len(ids),
            "labels": [-100] * len(p_ids) + ids[len(p_ids):]}

from datasets import Dataset
def build(msgs):
    rows = [r for r in (encode(m) for m in msgs) if r]
    print(f"{len(rows)}/{len(msgs)} examples kept")
    return Dataset.from_list(rows)
train_ds = build(train_msgs)
valid_ds = build(valid_msgs[:200])
test_ds = build(test_msgs[:400])


In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq, TrainerCallback

class TimeGuard(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        if (time.time() - START) / 3600 > TIME_LIMIT_H:
            print("time guard: stopping and saving before the Kaggle limit")
            control.should_training_stop = True
            control.should_save = True

# Real bf16 needs compute capability >= 8 (Ampere); the T4 (7.5) reports bf16 via emulation, so use fp16.
bf16 = torch.cuda.get_device_capability(0)[0] >= 8
import dataclasses
wanted = dict(
    output_dir=f"{OUT}/ckpt", num_train_epochs=1, per_device_train_batch_size=4, gradient_accumulation_steps=2,
    per_device_eval_batch_size=4, learning_rate=4e-5, warmup_steps=100,
    lr_scheduler_type="cosine_with_min_lr", lr_scheduler_kwargs={"min_lr_rate": 0.1},
    optim="adamw_8bit", weight_decay=0.0, max_grad_norm=1.0, fp16=not bf16, bf16=bf16,
    logging_steps=10, eval_strategy="steps", eval_steps=250, save_steps=250, save_total_limit=2,
    report_to="none", seed=0, dataloader_num_workers=2)
known = {f.name for f in dataclasses.fields(TrainingArguments)}
print("fp16" if not bf16 else "bf16", "| dropped (unsupported by this transformers):", [k for k in wanted if k not in known])
args = TrainingArguments(**{k: v for k, v in wanted.items() if k in known})
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=valid_ds,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
                  callbacks=[TimeGuard()])
print("baseline valid loss:", trainer.evaluate()["eval_loss"])
result = trainer.train()
print(result)


In [ ]:
ad = f"{OUT}/allam-tashkeel-v3-peft"
model.save_pretrained(ad)
tok.save_pretrained(ad)
test_loss = trainer.evaluate(test_ds)["eval_loss"]
log = {"train_runtime_h": round((time.time() - START) / 3600, 2), "unsloth": USE_UNSLOTH,
       "test_loss_400": test_loss, "test_ppl_400": math.exp(test_loss),
       "history": trainer.state.log_history, "base": REPO, "revision": REV, "gpu": torch.cuda.get_device_name(0)}
json.dump(log, open(f"{ad}/training_log.json", "w"), ensure_ascii=False, indent=1)
print({k: v for k, v in log.items() if k != "history"})
!cd {OUT} && zip -qr allam-tashkeel-v3-peft.zip allam-tashkeel-v3-peft && ls -la {OUT}


In [ ]:
# Sanity sample (the real evaluation runs on the Mac after conversion)
if USE_UNSLOTH:
    FastLanguageModel.for_inference(model)
msgs = test_msgs[0][:-1]
ids = tok.apply_chat_template(msgs, add_generation_prompt=False, return_tensors="pt").to(model.device)
out = model.generate(input_ids=ids, max_new_tokens=300, do_sample=True, temperature=0.7, top_p=0.8)
print(msgs[-1]["content"][:300]); print("-----"); print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
